# Stage A — build data artifacts (run ONCE, on any machine)

Produces the five files every training machine needs:

```
bpe8192-vocab.json   bpe8192-merges.txt
train.bin  train.bin.meta   val.bin  val.bin.meta
```

**Every run must use the byte-identical tokenizer and bins.** Retraining the BPE
on another machine gives a different vocab, and bits-per-byte stops being
comparable across your ablation rows.

When it finishes: **Save Version -> Save & Run All**, then publish
`/kaggle/working` as a Dataset (or attach this notebook's output directly as an
input to the Stage B notebooks).

Runtime ~20-30 min. CPU-only is fine — **turn the GPU off** to save quota.


In [ ]:
!pip install -q tokenizers datasets

In [ ]:
import os, glob, math, time, json
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

def resolve_paths():
    """DATA_DIR holds train.bin/val.bin/tokenizer (read-only). WORK holds outputs."""
    if os.path.isdir("/kaggle/working"):
        work = "/kaggle/working"
        cands = [d for d in glob.glob("/kaggle/input/*") if os.path.exists(f"{d}/train.bin")]
        data = cands[0] if cands else work
    elif os.path.isdir("/content"):
        work = "/content/work"; os.makedirs(work, exist_ok=True)
        data = work if os.path.exists(f"{work}/train.bin") else work
    else:
        work = os.path.abspath("./work"); os.makedirs(work, exist_ok=True); data = work
    return data, work

DATA_DIR, WORK = resolve_paths()
print("DATA_DIR:", DATA_DIR)
print("WORK    :", WORK)

In [ ]:
from datasets import load_dataset
from tokenizers import ByteLevelBPETokenizer

VOCAB_SIZE = 8192          # power of two -> tensor-core aligned lm_head
ds = load_dataset("roneneldan/TinyStories")
print(ds)

if not os.path.exists(f"{WORK}/bpe8192-vocab.json"):
    def corpus_iter(n=400_000):
        for i, r in enumerate(ds["train"]):
            if i >= n: break
            yield r["text"]
    t = ByteLevelBPETokenizer()
    t.train_from_iterator(corpus_iter(), vocab_size=VOCAB_SIZE, min_frequency=2,
                          special_tokens=["<|endoftext|>"])
    t.save_model(WORK, "bpe8192")

tok = ByteLevelBPETokenizer(f"{WORK}/bpe8192-vocab.json", f"{WORK}/bpe8192-merges.txt")
EOT = tok.token_to_id("<|endoftext|>")
assert tok.get_vocab_size() == VOCAB_SIZE, tok.get_vocab_size()
print("vocab", tok.get_vocab_size(), "eot", EOT)

In [ ]:
def build_split(split, out_bin):
    if os.path.exists(out_bin + ".meta"):
        return json.load(open(out_bin + ".meta"))
    rows, chunks, n_bytes, B = ds[split], [], 0, 5000
    for s in range(0, len(rows), B):
        texts = rows[s:s+B]["text"]
        for e in tok.encode_batch(texts):
            chunks.append(np.array(e.ids + [EOT], dtype=np.uint16))   # EOS per story
        n_bytes += sum(len(x.encode("utf-8")) for x in texts)
        if s % 200000 == 0: print(f"  {split} {s}/{len(rows)}")
    arr = np.concatenate(chunks); arr.tofile(out_bin)
    meta = {"n_tokens": int(arr.size), "n_bytes": int(n_bytes), "vocab_size": VOCAB_SIZE}
    json.dump(meta, open(out_bin + ".meta", "w")); return meta

tm = build_split("train", f"{WORK}/train.bin")
vm = build_split("validation", f"{WORK}/val.bin")
print(tm); print(vm)
print(f"\ntrain tokens {tm['n_tokens']/1e6:.1f}M  ({tm['n_tokens']*2/1e9:.2f} GB as uint16)")
print(f"val tokens/byte {vm['n_tokens']/vm['n_bytes']:.4f}   <- BPB conversion factor")

In [ ]:
# Fingerprint. Print this and check it matches on every training machine.
import hashlib
h = hashlib.sha256()
for f in ["bpe8192-vocab.json", "bpe8192-merges.txt"]:
    h.update(open(f"{WORK}/{f}", "rb").read())
a = np.memmap(f"{WORK}/train.bin", dtype=np.uint16, mode="r")
h.update(a[:1_000_000].tobytes()); h.update(a[-1_000_000:].tobytes())
print("DATA FINGERPRINT:", h.hexdigest()[:16])
print("files:", sorted(os.listdir(WORK)))